# Submit the managed AutoML experiment

[Guide](README.md) · [00: prepare data](00_prepare_data.ipynb) · [01: experiment](01_experiment.ipynb) · [02: results](02_results.ipynb) · [03: predictor](03_predictor.ipynb)

Complete [00 — Prepare data](00_prepare_data.ipynb) first. This notebook submits the installed GA managed pipeline, with the existing Connection Secret for training and test data. It does not compile a custom training pipeline or train AutoGluon locally.

In [1]:
from pathlib import Path
import sys

# Works when Jupyter starts in either the repo root or automl/.
automl_dir = Path.cwd() if Path('workflow.py').exists() else Path.cwd() / 'automl'
sys.path.insert(0, str(automl_dir.resolve()))
import pandas as pd
from IPython.display import display, Markdown, FileLink
import workflow as wf

settings = wf.settings()  # Never display this mapping: it may contain credentials.

In [2]:
manifest = wf.read('data_manifest.json')
arguments = wf.pipeline_arguments(manifest)
display(pd.Series(arguments, name='Pipeline inputs'))

train_data_secret_name                                                awss3
train_data_bucket_name                                        ai-eng-cracow
train_data_file_key       fraud-detection/automl/data/20261029T075342Z-8...
test_data_bucket_name                                         ai-eng-cracow
test_data_file_key        fraud-detection/automl/data/20261029T075342Z-8...
label_column                                                          fraud
task_type                                                            binary
positive_class                                                            1
eval_metric                                                         roc_auc
preset                                                                speed
top_n                                                                     1
Name: Pipeline inputs, dtype: object

## Start or resume

Default: `speed`, ROC-AUC, top one predictor. The pipeline handles preprocessing, model search and refit. Allow time beyond the selection budget for scheduling, refit and evaluation.

The cell resumes a saved run by default. To compare `balanced`, copy `outputs/` to a backup outside the repository, keeping `outputs/data_manifest.json` in place; change `.env`, refresh configuration and explicitly set `START_NEW_RUN=True`. Set it back to `False` afterwards. A failed preflight indicates an incompatible installed pipeline; do not omit `test_data_*` to work around it.

Set `RHOAI_DASHBOARD_URL` to the dashboard base URL to show Run details and AutoML results links. It is separate from the pipeline API endpoint in `KFP_ENDPOINT`.

In [3]:
client = wf.kfp_client()
START_NEW_RUN = False
run = wf.submit_experiment(client, manifest, new_run=START_NEW_RUN)
print('Run ID:', run['run_id'])
print('Pipeline version:', run['pipeline_version_id'])
print('Preset:', run['arguments']['preset'])
links = wf.run_links(run)
if links:
    for label, url in links.items():
        display(Markdown(f'[{label}]({url})'))
else:
    print('Set RHOAI_DASHBOARD_URL in automl/.env to show Run details and AutoML results links.')

/Users/mszewczy/projects/fraud-detection/.venv/lib/python3.13/site-packages/kfp/client/client.py:159: FutureWarning: This client only works with Kubeflow Pipeline v2.0.0-beta.2 and later versions.
  warnings.warn(


Run ID: 7c7e79be-552d-4959-9827-88e564970698
Pipeline version: fc470922-197f-4365-8652-7d9233ce6c34
Preset: speed


[Run details](https://rh-ai.apps.rosa.ai-eng-prod.wrh7.p3.openshiftapps.com/develop-train/pipelines/runs/mszewczy/runs/7c7e79be-552d-4959-9827-88e564970698)

[AutoML results](https://rh-ai.apps.rosa.ai-eng-prod.wrh7.p3.openshiftapps.com/develop-train/automl/results/mszewczy/7c7e79be-552d-4959-9827-88e564970698)

## Monitor the existing run

This cell prints progress and saves status. Interrupting it or hitting the local timeout does not cancel training. Run it again to resume monitoring. Inspect failed task logs and pod termination reasons to distinguish an OOM from other failures.

In [4]:
run = wf.monitor_experiment(client)
display(pd.Series({key: run.get(key) for key in
                   ['run_id', 'state', 'submitted_at', 'duration_seconds']}))

08:39:24 7c7e79be-552d-4959-9827-88e564970698: SUCCEEDED


run_id              7c7e79be-552d-4959-9827-88e564970698
state                                          SUCCEEDED
submitted_at            2026-10-29T07:54:19.151165+00:00
duration_seconds                                  2381.0
dtype: object

### Optional: prepare a ticket comment after failure

If monitoring failed, run `print(wf.ticket_report())` in a new cell to capture the run and dataset evidence. It leaves scores and serving status pending. Add observed pod/log evidence yourself.

Next: [02_results.ipynb](02_results.ipynb)